In [2]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Define paths
base_directory = 'Dataset'  # This directory should contain 'stroke' and 'no_stroke' folders
stroke_dir = os.path.join(base_directory, 'stroke_data')
no_stroke_dir = os.path.join(base_directory, 'noStroke_data')

# Collect image paths
stroke_images = [os.path.join(stroke_dir, fname) for fname in os.listdir(stroke_dir)]
no_stroke_images = [os.path.join(no_stroke_dir, fname) for fname in os.listdir(no_stroke_dir)]

# Create labels
stroke_labels = [1] * len(stroke_images)
no_stroke_labels = [0] * len(no_stroke_images)

# Combine data
x = stroke_images + no_stroke_images
y = stroke_labels + no_stroke_labels

# Split into train+val and test sets
x_temp, x_test, y_temp, y_test = train_test_split(x, y, test_size=0.1, stratify=y, random_state=42)

# Split train+val into train and validation sets
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42)

# Function to preprocess data for a CNN
def preprocess_images(image_paths, image_labels, batch_size=32):
    def load_and_process_image(image_path):
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, [224, 224])
        image /= 255.0  # Normalize to [0, 1] range
        return image
    
    # Create a dataset
    images = tf.data.Dataset.from_tensor_slices(image_paths)
    images = images.map(load_and_process_image, num_parallel_calls=tf.data.AUTOTUNE)
    labels = tf.data.Dataset.from_tensor_slices(image_labels)
    dataset = tf.data.Dataset.zip((images, labels))
    dataset = dataset.shuffle(buffer_size=1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset
    #dataset = dataset.shuffle(buffer_size=1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# Prepare datasets
train_dataset = preprocess_images(x_train, y_train)
val_dataset = preprocess_images(x_val, y_val)
test_dataset = preprocess_images(x_test, y_test)

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),  # Define the input shape explicitly
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print(model.summary())

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)

model.save('stroke_cnn_model.h5')

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 111, 111, 32)     0         
 2D)                                                             
                                                                 
 conv2d_4 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_4 (MaxPooling  (None, 54, 54, 64)       0         
 2D)                                                             
                                                                 
 conv2d_5 (Conv2D)           (None, 52, 52, 128)       73856     
                                                                 
 max_pooling2d_5 (MaxPooling  (None, 26, 26, 128)     

2024-04-20 17:13:31.564766: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [2714]
	 [[{{node Placeholder/_0}}]]
2024-04-20 17:13:31.564967: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [2714]
	 [[{{node Placeholder/_0}}]]


85/85 [==============================] - ETA: 0s - loss: 0.6566 - accuracy: 0.6470

2024-04-20 17:14:09.660495: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [679]
	 [[{{node Placeholder/_0}}]]
2024-04-20 17:14:09.660671: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_4' with dtype int32 and shape [679]
	 [[{{node Placeholder/_4}}]]


85/85 [==============================] - 41s 478ms/step - loss: 0.6566 - accuracy: 0.6470 - val_loss: 0.4866 - val_accuracy: 0.6657
Epoch 2/10
85/85 [==============================] - 43s 500ms/step - loss: 0.4124 - accuracy: 0.8150 - val_loss: 0.3495 - val_accuracy: 0.8409
Epoch 3/10
85/85 [==============================] - 47s 551ms/step - loss: 0.3294 - accuracy: 0.8541 - val_loss: 0.4526 - val_accuracy: 0.7761
Epoch 4/10
85/85 [==============================] - 43s 506ms/step - loss: 0.2022 - accuracy: 0.9237 - val_loss: 0.2021 - val_accuracy: 0.9219
Epoch 5/10
85/85 [==============================] - 43s 499ms/step - loss: 0.1135 - accuracy: 0.9576 - val_loss: 0.2544 - val_accuracy: 0.9278
Epoch 6/10
85/85 [==============================] - 43s 500ms/step - loss: 0.1017 - accuracy: 0.9639 - val_loss: 0.1201 - val_accuracy: 0.9720
Epoch 7/10
85/85 [==============================] - 43s 499ms/step - loss: 0.0323 - accuracy: 0.9912 - val_loss: 0.1543 - val_accuracy: 0.9661
Epoch 8/10

In [3]:
model.save('stroke_cnn_model.keras')